# 01 — HC60 hand-crafted features (v2)

Независимый блок из **60** признаков (`hc60_*`). Без TF-IDF, без legacy `hc_*`.

Спецификация: `v2/docs/hc60_feature_spec.md`. Код: `v2/src/core_hc60_features.py`.

Сохраняет `v2/data/interim/features/core_hc60_v2.parquet` + `core_hc60_v2_manifest.json`.

In [1]:
from __future__ import annotations
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

def resolve_v2() -> Path:
    cur = Path.cwd().resolve()
    for p in [cur, *cur.parents]:
        c = p / "v2" / "pyproject.toml"
        if c.is_file():
            return p / "v2"
        if p.name == "v2" and (p / "pyproject.toml").is_file():
            return p
    raise FileNotFoundError("Run from repo with v2/pyproject.toml")

BASE = resolve_v2()
sys.path.insert(0, str(BASE))
from src.core_hc60_features import extract_hc60, HC60_FEATURE_NAMES

ASSEMBLED = BASE / "data" / "interim" / "assembled"
OUT_DIR = BASE / "data" / "interim" / "features"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PQ = OUT_DIR / "core_hc60_v2.parquet"
OUT_M = OUT_DIR / "core_hc60_v2_manifest.json"

SOURCES = [
    (ASSEMBLED / "core_train.jsonl", None),
    (ASSEMBLED / "core_val.jsonl", None),
    (ASSEMBLED / "core_test_non_claude.jsonl", None),
    (ASSEMBLED / "core_test_claude_only.jsonl", None),
    (ASSEMBLED / "core_test_claude_binary.jsonl", None),
]

def load_jsonl(path: Path):
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)

rows = []
for path, _ in SOURCES:
    if not path.is_file():
        raise FileNotFoundError(path)
    for r in tqdm(list(load_jsonl(path)), desc=path.name):
        feats = extract_hc60(str(r.get("text") or ""))
        meta_keys = [
            "label", "split", "core_eval_slice", "scenario_family", "channel",
            "fraudness", "length_bin", "time_band", "source_family", "generator_lane",
            "origin_model", "text",
        ]
        row = {k: r.get(k) for k in meta_keys}
        row.update(feats)
        row["_source_path"] = str(path.relative_to(BASE))
        rows.append(row)

df = pd.DataFrame(rows)
df.to_parquet(OUT_PQ, index=False)
manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "random_seed_note": "feature extraction deterministic; assembly seed=42 in Core",
    "n_rows": len(df),
    "n_features": len(HC60_FEATURE_NAMES),
    "feature_names": list(HC60_FEATURE_NAMES),
    "parquet": str(OUT_PQ.resolve()),
}
OUT_M.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("Wrote", OUT_PQ, "rows", len(df))
print("Manifest", OUT_M)


core_train.jsonl:   0%|          | 0/13554 [00:00<?, ?it/s]

core_val.jsonl:   0%|          | 0/2706 [00:00<?, ?it/s]

core_test_non_claude.jsonl:   0%|          | 0/1816 [00:00<?, ?it/s]

core_test_claude_only.jsonl:   0%|          | 0/2059 [00:00<?, ?it/s]

core_test_claude_binary.jsonl:   0%|          | 0/3692 [00:00<?, ?it/s]

Wrote /Users/askar/projects/antifraud-deepfake-detection/v2/data/interim/features/core_hc60_v2.parquet rows 23827
Manifest /Users/askar/projects/antifraud-deepfake-detection/v2/data/interim/features/core_hc60_v2_manifest.json
